# Lab4: 多轮迭代式框+文本提示分割
## Iterative Box + Text Prompt Segmentation with Accumulated Masks

---

### 实验背景

Lab2 证明了 **文本+框提示的组合方案** 分割效果最好，但每次只能单轮推理。

Lab4 在 Lab2 的基础上增加了：

1. **多轮分割**：同一张图上可以反复添加新的框+文本提示，结果**累积**显示
2. **掩码管理**：每个掩码都有 **可见/隐藏开关** 和 **删除按钮**
3. **合成可视化**：所有可见掩码以不同颜色叠加显示，并标注分数
4. **批量导出**：一键导出所有可见掩码（PNG）+ 合成图 + 元数据

### 操作方法

1. 选择图片 → 输入文本提示 → 调整框 → 点击 Run Segmentation
2. 重复上述步骤，可对同一张图添加多种目标的分割
3. 在掩码管理列表中取消勾选可隐藏错误掩码，点击 删除 彻底删除
4. 调整好后点击 Export All Visible 导出结果

In [3]:
# ========== 环境准备 ==========
import os, sys, csv, json, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

from IPython.display import display
import ipywidgets as widgets

# 修复 matplotlib 中文
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun',
                                     'FangSong', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline
print("环境就绪 ✅")

环境就绪 ✅


In [4]:
# ========== 路径配置 ==========
_cwd = Path.cwd().resolve()
if (_cwd / "sam3").is_dir():
    REPO_ROOT = _cwd
elif (_cwd.parent / "sam3").is_dir():
    REPO_ROOT = _cwd.parent
else:
    raise RuntimeError(f"Cannot find repo root from {_cwd}")

CKPT_PATH = REPO_ROOT / "sam3.pt"
INPUT_DIR = REPO_ROOT / "Inputs" / "RawImages"
CSV_PATH = INPUT_DIR / "试标注图像清单.csv"
LAB4_OUTPUT = REPO_ROOT / "Outputs" / "Lab4_output"
LAB4_OUTPUT.mkdir(parents=True, exist_ok=True)

if not CKPT_PATH.exists():
    raise FileNotFoundError(f"权重未找到: {CKPT_PATH}")

# 读取 CSV
image_notes = {}
if CSV_PATH.exists():
    with open(CSV_PATH, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_notes[row["编号"]] = row

# 图片列表
image_files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith("_RawImage.png")])
print(f"图片数量: {len(image_files)}")
print(f"输出目录: {LAB4_OUTPUT}")
print(f"权重文件: {CKPT_PATH}")

图片数量: 16
输出目录: C:\Users\Legion\Desktop\WIE\SAM3_Labs\Outputs\Lab4_output
权重文件: C:\Users\Legion\Desktop\WIE\SAM3_Labs\sam3.pt


In [5]:
# ========== 加载 SAM 3 模型 ==========
print("[⏳] 正在加载 SAM 3 模型...")
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"    设备: {device}")

model = build_sam3_image_model(
    checkpoint_path=str(CKPT_PATH),
    load_from_HF=False,
    device=device,
    eval_mode=True,
)
processor = Sam3Processor(model, device=device)
print(f"[✅] SAM 3 加载完成 | 参数: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

[⏳] 正在加载 SAM 3 模型...
    设备: cuda
[✅] SAM 3 加载完成 | 参数: 841M


In [ ]:
# ========== Lab4: 多轮迭代式框+文本提示分割 ==========

# ---- 全局状态 ----
_current_img = None
_current_img_id = None
_current_img_w = 800
_current_img_h = 600
seg_entries = []       # {id, text, box, mask, score, visible}
next_entry_id = 0
_cmap = plt.cm.tab10


def _get_color(entry_id):
    return _cmap(entry_id % 10)


def load_image(img_id):
    global _current_img, _current_img_id, _current_img_w, _current_img_h
    global seg_entries, next_entry_id

    filename = f"{img_id}_RawImage.png"
    img_path = INPUT_DIR / filename
    if not img_path.exists():
        return

    _current_img = Image.open(img_path).convert("RGB")
    _current_img_id = img_id
    _current_img_w, _current_img_h = _current_img.size

    # 重置状态
    seg_entries = []
    next_entry_id = 0

    # 更新滑块范围
    slider_x2.max = _current_img_w
    slider_y2.max = _current_img_h
    slider_x1.max = max(0, _current_img_w - 10)
    slider_y1.max = max(0, _current_img_h - 10)

    # 显示描述
    notes = image_notes.get(img_id, {})
    desc_output.clear_output()
    with desc_output:
        print(f"📷 {filename} ({_current_img_w}x{_current_img_h})")
        for k in ["主要构件", "场景特点", "干扰项"]:
            val = notes.get(k, "")
            if val:
                print(f"  - {val}")

    update_preview(None)
    rebuild_mask_list()
    redraw_visualization()


def update_preview(change):
    if _current_img is None:
        return
    x1, y1 = slider_x1.value, slider_y1.value
    x2, y2 = slider_x2.value, slider_y2.value
    preview_output.clear_output(wait=True)
    with preview_output:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(np.array(_current_img))
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                             linewidth=2, edgecolor='lime', facecolor='lime', alpha=0.2)
        ax.add_patch(rect)
        cx = ((x1+x2)/2)/_current_img_w
        cy = ((y1+y2)/2)/_current_img_h
        bw = (x2-x1)/_current_img_w
        bh = (y2-y1)/_current_img_h
        ax.set_title(f"Norm: cx={cx:.3f}, cy={cy:.3f}, w={bw:.3f}, h={bh:.3f}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()


def run_segmentation(btn):
    global seg_entries, next_entry_id
    if _current_img is None:
        return
    text_prompt = text_input.value.strip()
    if not text_prompt:
        print("请输入文本提示")
        return
    x1, y1 = slider_x1.value, slider_y1.value
    x2, y2 = slider_x2.value, slider_y2.value
    cx = ((x1+x2)/2)/_current_img_w
    cy = ((y1+y2)/2)/_current_img_h
    bw = (x2-x1)/_current_img_w
    bh = (y2-y1)/_current_img_h
    result_output.clear_output(wait=True)
    with result_output:
        print(f"[⏳] Running... text='{text_prompt}'")
    try:
        state = processor.set_image(_current_img)
        state = processor.set_text_prompt(prompt=text_prompt, state=state)
        state = processor.add_geometric_prompt(
            box=[cx, cy, bw, bh], label=True, state=state)
        masks, scores = state["masks"], state["scores"]
        new_count = 0
        for i in range(len(masks)):
            seg_entries.append({
                "id": next_entry_id,
                "text": text_prompt,
                "box": (x1, y1, x2, y2),
                "mask": masks[i].squeeze().cpu(),
                "score": scores[i].item(),
                "visible": True,
            })
            next_entry_id += 1
            new_count += 1
        result_output.clear_output(wait=True)
        with result_output:
            print(f"[✅] 新增 {new_count} 个掩码 | 总计 {len(seg_entries)} 个掩码")
        rebuild_mask_list()
        redraw_visualization()
    except Exception as e:
        import traceback; traceback.print_exc()
        result_output.clear_output(wait=True)
        with result_output:
            print(f"[❌] 推理失败: {e}")


def toggle_mask_visibility(entry_id, change):
    for entry in seg_entries:
        if entry["id"] == entry_id:
            entry["visible"] = change.new
            break
    redraw_visualization()


def delete_mask(entry_id, btn=None):
    global seg_entries
    seg_entries = [e for e in seg_entries if e["id"] != entry_id]
    rebuild_mask_list()
    redraw_visualization()


def clear_all(btn):
    global seg_entries, next_entry_id
    seg_entries = []; next_entry_id = 0
    rebuild_mask_list(); redraw_visualization()
    result_output.clear_output()
    with result_output:
        print("已清除所有掩码")


# ---- 动态掩码管理列表 ----
mask_list_container = widgets.VBox([])

def rebuild_mask_list():
    children = []
    for entry in seg_entries:
        eid = entry["id"]
        cb = widgets.Checkbox(value=entry["visible"], description=f"#{eid:02d}")
        cb.observe(lambda c, _eid=eid: toggle_mask_visibility(_eid, c), names="value")
        c = _get_color(eid)
        hex_c = f"#{int(c[0]*255):02x}{int(c[1]*255):02x}{int(c[2]*255):02x}"
        info = widgets.HTML(
            f"<span style='font-size:10pt'>score={entry['score']:.3f} | "
            f"<b>{entry['text']}</b> | "
            f"box=({entry['box'][0]},{entry['box'][1]})-({entry['box'][2]},{entry['box'][3]})</span>")
        del_btn = widgets.Button(description="🗑",
            button_style="danger", layout={"width": "36px", "height": "28px"})
        del_btn.on_click(lambda btn, _eid=eid: delete_mask(_eid))
        dot = widgets.HTML(f"<span style='display:inline-block;width:14px;height:14px;"
            f"background:{hex_c};border-radius:3px;'></span>")
        children.append(widgets.HBox([dot, cb, info, del_btn]))
    mask_list_container.children = children


# ---- 重绘可视化 ----
vis_output = widgets.Output()

def redraw_visualization():
    vis_output.clear_output(wait=True)
    if _current_img is None:
        return
    with vis_output:
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(np.array(_current_img))
        overlay = np.zeros((_current_img_h, _current_img_w, 4), dtype=np.float32)
        for entry in seg_entries:
            if not entry["visible"]:
                continue
            mask = entry["mask"].numpy()
            color = _get_color(entry["id"])
            overlay[mask] = color[:3] + (0.5,)
            ys, xs = np.where(mask)
            if len(xs) > 0:
                cy_m, cx_m = ys[len(ys)//2], xs[len(xs)//2]
                ax.text(cx_m, cy_m, f"#{entry['id']}:{entry['score']:.2f}",
                    color='white', fontsize=8, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='black', alpha=0.6))
        for entry in seg_entries:
            if entry["visible"]:
                x1, y1, x2, y2 = entry["box"]
                color = _get_color(entry["id"])
                ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                    linewidth=1.5, edgecolor=color, facecolor='none', linestyle='--'))
        ax.imshow(overlay)
        visible = sum(1 for e in seg_entries if e["visible"])
        ax.set_title(f"可见掩码: {visible}/{len(seg_entries)} | 图片: {_current_img_id}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()


# ---- 导出 ----
def export_all(btn):
    if _current_img is None or not seg_entries:
        print("没有掩码可导出")
        return
    ts = datetime.now().strftime("%H%M%S")
    save_dir = LAB4_OUTPUT / f"{_current_img_id}_{ts}"
    save_dir.mkdir(parents=True, exist_ok=True)
    vis = [e for e in seg_entries if e["visible"]]
    for entry in vis:
        mask_np = (entry["mask"].numpy() * 255).astype(np.uint8)
        fname = f"mask_{entry['id']:02d}_{entry['score']:.3f}_{entry['text'].replace(' ','_')}.png"
        Image.fromarray(mask_np).save(save_dir / fname)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(np.array(_current_img))
    overlay = np.zeros((_current_img_h, _current_img_w, 4), dtype=np.float32)
    for entry in vis:
        mask = entry["mask"].numpy()
        overlay[mask] = _get_color(entry["id"])[:3] + (0.5,)
    ax.imshow(overlay)
    ax.set_title(f"Lab4 - {_current_img_id} | {len(vis)} masks")
    ax.axis("off")
    plt.tight_layout()
    fig.savefig(save_dir / "composite.png", dpi=150, bbox_inches="tight")
    plt.close()
    meta = [{"id": e["id"], "text": e["text"], "box": list(e["box"]), "score": e["score"]} for e in vis]
    with open(save_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    result_output.clear_output(wait=True)
    with result_output:
        print(f"[✅] 导出 {len(vis)} 个掩码到: {save_dir}")


# ========== 构建 UI 控件 ==========

img_selector = widgets.Dropdown(
    options=[(f.replace("_RawImage.png",""), f.replace("_RawImage.png","")) for f in image_files],
    description="选择图片:", layout={"width": "200px"})

text_input = widgets.Text(
    value="swallowtail ridge",
    description="文本提示:", layout={"width": "400px"})

slider_x1 = widgets.IntSlider(value=100, min=0, max=700, step=1,
    description="x1(left):", layout={"width": "500px"})
slider_y1 = widgets.IntSlider(value=100, min=0, max=500, step=1,
    description="y1(top):", layout={"width": "500px"})
slider_x2 = widgets.IntSlider(value=300, min=10, max=800, step=1,
    description="x2(right):", layout={"width": "500px"})
slider_y2 = widgets.IntSlider(value=300, min=10, max=600, step=1,
    description="y2(bottom):", layout={"width": "500px"})

run_btn = widgets.Button(description="▶ Run Segmentation",
    button_style="success", layout={"width": "200px", "height": "36px"})
run_btn.on_click(run_segmentation)

clear_btn = widgets.Button(description="🗑 Clear All",
    button_style="danger", layout={"width": "120px", "height": "36px"})
clear_btn.on_click(clear_all)

export_btn = widgets.Button(description="💾 Export All",
    button_style="primary", layout={"width": "160px", "height": "36px"})
export_btn.on_click(export_all)

desc_output = widgets.Output()
preview_output = widgets.Output()
result_output = widgets.Output()

# ---- 布局 ----
controls = widgets.VBox([
    widgets.HBox([img_selector, text_input]),
    widgets.HBox([slider_x1, slider_x2]),
    widgets.HBox([slider_y1, slider_y2]),
    widgets.HBox([run_btn, clear_btn, export_btn]),
    desc_output,
])

# ---- 绑定事件 ----
img_selector.observe(lambda c: load_image(c.new) if c.new else None, names="value")
slider_x1.observe(update_preview, names="value")
slider_y1.observe(update_preview, names="value")
slider_x2.observe(update_preview, names="value")
slider_y2.observe(update_preview, names="value")

# ---- 显示 ----
display(controls)
display(widgets.HTML("<hr><b>框预览:</b>"))
display(preview_output)
display(widgets.HTML("<hr><b>分割结果:</b>"))
display(result_output)
display(widgets.HTML("<hr><b>掩码管理 (勾选=可见, 删除):</b>"))
display(mask_list_container)
display(widgets.HTML("<hr><b>累积掩码可视化:</b>"))
display(vis_output)

# 加载第一张图
if image_files:
    first_id = image_files[0].replace("_RawImage.png", "")
    load_image(first_id)

HTML(value='<hr><b>框预览:</b>')

Output()

HTML(value='<hr><b>分割结果:</b>')

Output()

HTML(value='<hr><b>掩码管理 (勾选=可见, 删除):</b>')

VBox()

HTML(value='<hr><b>累积掩码可视化:</b>')

Output()

### 推荐提示

| 建筑特征 | 提示词 | 框位置 |
|-----------|-------------|----------|
| Swallowtail ridge | `swallowtail ridge` | 屋脊两端 |
| Horse-back gable | `horse-back gable` | 侧墙顶部 |
| Porcelain inlay | `porcelain inlay` | 屋顶装饰 |
| Dougong bracket | `dougong bracket` / `wooden bracket` | 斗拱位置 |
| Red brick wall | `red brick wall` | 红砖墙体 |
| Stone door frame | `stone door frame` | 门框 |
| Waterwheel frieze | `waterwheel frieze` | 廊檐彩绘 |
| Roof ridge sculpture | `roof ridge sculpture` | 屋脊雕塑 |

> 提示：可以对同一张图多次运行，每次用不同的文本+框提示分割不同的建筑特征。错误的掩码可以取消勾选或点击删除。